In [ ]:
from model import SchemaIntegrationRuns
from utils import CACHE_DIR
from evaluation import SchemaIntegrationEvaluation
import os
from dotenv import dotenv_values
from langchain.chat_models import init_chat_model
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
# Initialize LLM
model_name = "gpt-5.2-2025-12-11"
# model_name = "Qwen/Qwen3.6-27B"
reasoning = "None"
model_type = "openai"
# model_type = "hf" # For models loaded through HuggingFace

if model_type == "hf":
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype="float16"
    )
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=CACHE_DIR)
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16, low_cpu_mem_usage=True, device_map="auto", cache_dir=CACHE_DIR, quantization_config=quant_config)
    if "qwen" in model_name.lower():
        # Update chat format to fix issues with tool calls
        with open("preprocessing/qwen_chat_template.jinja", "r") as f:
            tokenizer.chat_template = f.read()
else:
    config = dotenv_values("../key.env")
    os.environ['OPENAI_API_KEY'] = config["OPENAI_API_KEY"]
    OPENAI_API_KEY = config["OPENAI_API_KEY"]
    
    if reasoning != "None":
        model = init_chat_model(openai_api_key=OPENAI_API_KEY, temperature=1, model=model_name, reasoning = {"effort": reasoning, "summary": "auto"})
    else:
        model = init_chat_model(openai_api_key=OPENAI_API_KEY, temperature=1, model=model_name)
    tokenizer = None

### Run all workflows

In [ ]:
prompt_name = "long-prompt-updated-4"
benchmark = "SINT-Benchmark"
for folder_name in os.listdir(f"data/selected-tables/{benchmark}/"):
    for sequence_of_phases in [
        ["table_splitting_phase", "schema_matching_phase", "grouping_phase", "attribute_merging_phase", "final_integration_phase"],
        # ["schema_matching_phase", "attribute_merging_phase", "table_splitting_phase", "final_integration_phase"],
        # ["table_splitting_phase", "grouping_phase", "attribute_merging_phase", "final_integration_phase"]

    ]:
        # Run config
        config = {
            "benchmark": benchmark,
            "folder_name": folder_name,
            "prompt_name": prompt_name,
            "table_path": f"data/selected-tables/{benchmark}/{folder_name}/",
            "table_format": "markdown",
            "table_rows": 10, # For Qwen we use 5
            "model": model,
            "model_type": model_type,
            "model_name": model_name,
            "reasoning": reasoning,
            "tokenizer": tokenizer,
            "self_consistency": True,
            # "self_consistency": False,
            "schema_matching_batch_size": 1,
            "merging_batch_size": "all", 
            "num_runs": 3,
            "sequence_of_phases": sequence_of_phases,
            # "logfile_name": "logs/" # to continue from a previous run, specify the log file name found in logs folder
        }
        schema_integration_workflow = SchemaIntegrationRuns(**config)
        if not config.get("logfile_name"):
            schema_integration_workflow.initialize_workflow()
        else:
            schema_integration_workflow.initialize_workflow_from_file()
        
        schema_integration_workflow.run_workflow()

### Manual workflow runs

In [ ]:
prompt_name = "long-prompt-updated-4"
folder_name = "goby_selected" # choose use-case
config = {
    "benchmark": benchmark,
    "folder_name": folder_name,
    "prompt_name": prompt_name,
    "table_path": f"data/selected-tables/{benchmark}/{folder_name}/",
    "table_format": "markdown",
    "table_rows": 10, # For Qwen we use 5
    "model": model,
    "model_type": model_type,
    "model_name": model_name,
    "reasoning": reasoning,
    "tokenizer": tokenizer,
    "self_consistency": True,
    # "self_consistency": False,
    "schema_matching_batch_size": 1,
    "merging_batch_size": "all", 
    "num_runs": 3,
    "sequence_of_phases": sequence_of_phases,
    # "logfile_name": "logs/" # to continue from a previous run, specify the log file name found in logs folder
}

In [ ]:
schema_integration_workflow = SchemaIntegrationRuns(**config)
schema_integration_workflow.initialize_workflow()

In [ ]:
phase_name = "table_splitting_phase"
phase_function = schema_integration_workflow.get_function(phase_name=phase_name, run_nr=0)["function"]
phase_function(run_feedback_loop=True, fds=schema_integration_workflow.fds, perfect_case=False)

# Postprocessing
phase_postprocessing_function = schema_integration_workflow.get_function(phase_name=phase_name, run_nr=0)["postprocessing_function"]
if phase_postprocessing_function:
    phase_postprocessing_function()

In [ ]:
phase_name = "schema_matching_phase"
phase_function = schema_integration_workflow.get_function(phase_name=phase_name, run_nr=0)["function"]
phase_function(run_feedback_loop=True, fds=schema_integration_workflow.fds, schema_matching_batch_size=3)

# Postprocessing
phase_postprocessing_function = schema_integration_workflow.get_function(phase_name=phase_name, run_nr=0)["postprocessing_function"]
if phase_postprocessing_function:
    phase_postprocessing_function()

In [ ]:
phase_name = "grouping_phase"
phase_function = schema_integration_workflow.get_function(phase_name=phase_name, run_nr=0)["function"]
phase_function(run_feedback_loop=True, fds=schema_integration_workflow.fds, perfect_case=False)

# Postprocessing
phase_postprocessing_function = schema_integration_workflow.get_function(phase_name=phase_name, run_nr=0)["postprocessing_function"]
if phase_postprocessing_function:
    phase_postprocessing_function()

In [ ]:
phase_name = "attribute_merging_phase"
phase_function = schema_integration_workflow.get_function(phase_name=phase_name, run_nr=0)["function"]
phase_function(run_feedback_loop=True, fds=schema_integration_workflow.fds, perfect_case=False, merging_batch_size="all")
# Postprocessing
phase_postprocessing_function = schema_integration_workflow.get_function(phase_name=phase_name, run_nr=0)["postprocessing_function"]
if phase_postprocessing_function:
    phase_postprocessing_function()

In [ ]:
phase_name = "final_integration_phase"
phase_function = schema_integration_workflow.get_function(phase_name=phase_name, run_nr=0)["function"]
phase_function(run_feedback_loop=True, fds=schema_integration_workflow.fds, perfect_case=False)

# Postprocessing
phase_postprocessing_function = schema_integration_workflow.get_function(phase_name=phase_name, run_nr=0)["postprocessing_function"]
if phase_postprocessing_function:
    phase_postprocessing_function()